In this notebook we will go over the basics of creating Kueue enabled resources with the SDK

In [ ]:
# Import pieces from codeflare-sdk
from codeflare_sdk import Cluster, ClusterConfiguration, TokenAuthentication

In [ ]:
# Create authentication object for user permissions
# IF unused, SDK will automatically check for default kubeconfig, then in-cluster config
# KubeConfigFileAuthentication can also be used to specify kubeconfig path manually
auth = TokenAuthentication(
    token = "XXXXX",
    server = "XXXXX",
    skip_tls=False
)
auth.login()

In [ ]:
# Create and configure our cluster object
# The SDK will try to find the name of your default local queue based on the annotation "kueue.x-k8s.io/default-queue": "true"
cluster = Cluster(ClusterConfiguration(
    name='kueue-test',
    namespace='default',
    num_workers=2,
    min_cpus=1,
    max_cpus=1,
    min_memory=4,
    max_memory=4,
    num_gpus=0,
    image="quay.io/project-codeflare/ray:latest-py39-cu118",
    # local_queue="local-queue-name" # Specify the local queue manually
))

In [ ]:
cluster.up()

In [ ]:
cluster.wait_ready()

In [ ]:
cluster.status()

In [ ]:
cluster.details()

In [ ]:
cluster.down()

Now, we will submit Jobs directly to Kueue, which will schedule a Batch Job to run with the requested resources using the Kueue Torchx scheduler:

NOTE: To test this demo in an air-gapped/ disconnected environment alter the training script to use a local dataset.

In [ ]:
from codeflare_sdk import DDPJobDefinition
jobdef = DDPJobDefinition(
    name="mnistjob",
    script="mnist.py",
    # script="mnist_disconnected.py", # training script for disconnected environment
    scheduler_args={"namespace": "default"},
    j="1x1",
    gpu=0,
    cpu=1,
    memMB=8000,
    image="quay.io/project-codeflare/mnist-job-test:v0.0.1",
)
job = jobdef.submit()

In [ ]:
job.status()

In [ ]:
job.logs()

This time, once the pods complete, we can clean them up alongside any other associated resources. The following command can also be used to delete jobs early for Kueue submission:

In [ ]:
job.cancel()

In [ ]:
auth.logout()